In [32]:
import datetime
import logging
import pathlib

import polars as pl
from memory_profiler import profile
from polars import Schema
from polars.datatypes import (
    Boolean,
    Float64,
    Int64,
    List,
    String,
    Struct,
)
import polars.selectors as cs

ROOT_DIR_PATH = pathlib.Path(".").resolve().parent
DATA_DIR_PATH = ROOT_DIR_PATH / "data"
FAKE_GAMING_DATA_DIR = DATA_DIR_PATH / "fake_gaming_data"

## Define Schema

In [4]:
schema = Schema(
    {
        "team_id": String,
        "name": String,
        "created_date": String,
        "ranking": Int64,
        "total_winnings": Int64,
        "members": List(
            Struct(
                {
                    "player_id": String,
                    "username": String,
                    "account_details": Struct(
                        {
                            "email": String,
                            "registration_date": String,
                            "premium_status": Boolean,
                            "country": String,
                            "language": String,
                        }
                    ),
                    "stats": Struct(
                        {
                            "level": Int64,
                            "experience": Int64,
                            "total_matches": Int64,
                            "win_rate": Float64,
                            "playtime_hours": Int64,
                            "achievements_completed": Int64,
                        }
                    ),
                    "inventory": Struct(
                        {
                            "currency": Struct({"premium": Int64, "standard": Int64}),
                            "items": List(
                                Struct(
                                    {
                                        "item_id": String,
                                        "name": String,
                                        "type": String,
                                        "rarity": String,
                                        "level_requirement": Int64,
                                        "stats": Struct(
                                            {
                                                "attack": Int64,
                                                "defense": Int64,
                                                "magic": Int64,
                                                "speed": Int64,
                                            }
                                        ),
                                    }
                                )
                            ),
                        }
                    ),
                    "achievements": List(
                        Struct(
                            {
                                "id": String,
                                "name": String,
                                "difficulty": String,
                                "completion_rate": Float64,
                                "points": Int64,
                                "date": String,
                            }
                        )
                    ),
                    "recent_matches": List(
                        Struct(
                            {
                                "match_id": String,
                                "game_mode": String,
                                "map": String,
                                "duration_minutes": Int64,
                                "date": String,
                                "stats": Struct(
                                    {
                                        "kills": Int64,
                                        "deaths": Int64,
                                        "assists": Int64,
                                        "damage_dealt": Int64,
                                        "healing_done": Int64,
                                        "accuracy": Float64,
                                        "headshot_percentage": Float64,
                                        "objectives_completed": Int64,
                                    }
                                ),
                                "rewards": Struct(
                                    {
                                        "experience": Int64,
                                        "currency": Int64,
                                        "items_dropped": List(
                                            Struct(
                                                {
                                                    "item_id": String,
                                                    "name": String,
                                                    "rarity": String,
                                                    "value": Int64,
                                                }
                                            )
                                        ),
                                    }
                                ),
                            }
                        )
                    ),
                }
            )
        ),
        "tournament_history": List(
            Struct(
                {
                    "tournament_id": String,
                    "name": String,
                    "placement": Int64,
                    "prize_money": Int64,
                    "matches_played": Int64,
                }
            )
        ),
    }
)

## Read LazyFrame 
(No execution triggered, data is loaded lazily aka. dag generation, like in spark)

In [5]:
lf: pl.LazyFrame = pl.scan_ndjson(
    FAKE_GAMING_DATA_DIR / "data.json",
    schema=schema,
)

In [6]:
lf.explain(optimized=True)

'NDJson SCAN [/Users/jakubpluta/Repositories/priv/polars-internals/data/fake_gaming_data/data.json]\nPROJECT */7 COLUMNS'

In [7]:
sample = lf.limit(5).collect(
    streaming=True
)  # read only 5 records. LazyFrame materialized into DataFrame

In [8]:
sample.head(1)

shape: (1, 7)
┌──────────────┬──────────────┬──────────────┬─────────┬──────────────┬──────────────┬─────────────┐
│ team_id      ┆ name         ┆ created_date ┆ ranking ┆ total_winnin ┆ members      ┆ tournament_ │
│ ---          ┆ ---          ┆ ---          ┆ ---     ┆ gs           ┆ ---          ┆ history     │
│ str          ┆ str          ┆ str          ┆ i64     ┆ ---          ┆ list[struct[ ┆ ---         │
│              ┆              ┆              ┆         ┆ i64          ┆ 7]]          ┆ list[struct │
│              ┆              ┆              ┆         ┆              ┆              ┆ [5]]        │
╞══════════════╪══════════════╪══════════════╪═════════╪══════════════╪══════════════╪═════════════╡
│ 3d11adbe-36d ┆ Squad        ┆ 2024-11-22   ┆ 493     ┆ 498804       ┆ [{"f16ba9c0- ┆ [{"33b25f51 │
│ 0-4af4-9ac1- ┆ Johnson Inc  ┆              ┆         ┆              ┆ f9ea-4079-b2 ┆ -088d-4014- │
│ 194ef7…      ┆              ┆              ┆         ┆              ┆ 06-a8e…      ┆ 8e23-06f…   │
└──────────────┴──────────────┴──────────────┴─────────┴──────────────┴──────────────┴─────────────┘

In [9]:
sample.schema

Schema([('team_id', String),
        ('name', String),
        ('created_date', String),
        ('ranking', Int64),
        ('total_winnings', Int64),
        ('members',
         List(Struct({'player_id': String, 'username': String, 'account_details': Struct({'email': String, 'registration_date': String, 'premium_status': Boolean, 'country': String, 'language': String}), 'stats': Struct({'level': Int64, 'experience': Int64, 'total_matches': Int64, 'win_rate': Float64, 'playtime_hours': Int64, 'achievements_completed': Int64}), 'inventory': Struct({'currency': Struct({'premium': Int64, 'standard': Int64}), 'items': List(Struct({'item_id': String, 'name': String, 'type': String, 'rarity': String, 'level_requirement': Int64, 'stats': Struct({'attack': Int64, 'defense': Int64, 'magic': Int64, 'speed': Int64})}))}), 'achievements': List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})), 'recent_matches': List(Struct(

In [10]:
sample.dtypes

[String,
 String,
 String,
 Int64,
 Int64,
 List(Struct({'player_id': String, 'username': String, 'account_details': Struct({'email': String, 'registration_date': String, 'premium_status': Boolean, 'country': String, 'language': String}), 'stats': Struct({'level': Int64, 'experience': Int64, 'total_matches': Int64, 'win_rate': Float64, 'playtime_hours': Int64, 'achievements_completed': Int64}), 'inventory': Struct({'currency': Struct({'premium': Int64, 'standard': Int64}), 'items': List(Struct({'item_id': String, 'name': String, 'type': String, 'rarity': String, 'level_requirement': Int64, 'stats': Struct({'attack': Int64, 'defense': Int64, 'magic': Int64, 'speed': Int64})}))}), 'achievements': List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})), 'recent_matches': List(Struct({'match_id': String, 'game_mode': String, 'map': String, 'duration_minutes': Int64, 'date': String, 'stats': Struct({'kills': Int64, 'dea

In [11]:
df = lf.collect()  # load whole data into memory

In [12]:
df.shape  # 14701 records, 7 columns

(14701, 7)

In [13]:
df.head(1)

shape: (1, 7)
┌──────────────┬──────────────┬──────────────┬─────────┬──────────────┬──────────────┬─────────────┐
│ team_id      ┆ name         ┆ created_date ┆ ranking ┆ total_winnin ┆ members      ┆ tournament_ │
│ ---          ┆ ---          ┆ ---          ┆ ---     ┆ gs           ┆ ---          ┆ history     │
│ str          ┆ str          ┆ str          ┆ i64     ┆ ---          ┆ list[struct[ ┆ ---         │
│              ┆              ┆              ┆         ┆ i64          ┆ 7]]          ┆ list[struct │
│              ┆              ┆              ┆         ┆              ┆              ┆ [5]]        │
╞══════════════╪══════════════╪══════════════╪═════════╪══════════════╪══════════════╪═════════════╡
│ 3d11adbe-36d ┆ Squad        ┆ 2024-11-22   ┆ 493     ┆ 498804       ┆ [{"f16ba9c0- ┆ [{"33b25f51 │
│ 0-4af4-9ac1- ┆ Johnson Inc  ┆              ┆         ┆              ┆ f9ea-4079-b2 ┆ -088d-4014- │
│ 194ef7…      ┆              ┆              ┆         ┆              ┆ 06-a8e…      ┆ 8e23-06f…   │
└──────────────┴──────────────┴──────────────┴─────────┴──────────────┴──────────────┴─────────────┘

In [17]:
### members, tournament_history  is a list of structs, we can explode them into multi rows

In [14]:
df_members_exploded = df.explode(pl.col("members"))

In [15]:
df_members_exploded.head()

team_id,name,created_date,ranking,total_winnings,members,tournament_history
str,str,str,i64,i64,struct[7],list[struct[5]]
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""Squad Johnson Inc""","""2024-11-22""",493,498804,"{""f16ba9c0-f9ea-4079-b206-a8e0b7737eaa"",""marvinthomas"",{""kevin49@example.com"",""2025-01-02"",true,""Gibraltar"",""ja""},{91,159523,313,50.35,1173,32},{{5067,85179},[{""90f6348c-3f55-46cd-8de4-66c3d1fc0ac9"",""Heavy Armor"",""Accessory"",""Epic"",87,{16,2,90,46}}, {""bf43fee5-9024-4742-bfc9-d9920efe82c8"",""Dragon Sword"",""Accessory"",""Legendary"",10,{72,29,20,74}}, … {""3f715010-385a-449b-bfc5-be92c6255517"",""Mage Staff"",""Accessory"",""Rare"",79,{94,38,54,21}}]},[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}],[{""7f06e144-0d2c-439f-9c98-7a0354f26ef3"",""Ranked"",""Desert Temple"",16,""2024-11-16"",{6,2,15,44202,2655,62.51,28.01,2},{515,231,[{""9e234564-ea7c-490f-993c-6242f47adf23"",""Epic Shield"",""Common"",4508}]}}, {""6643172c-a3b8-47af-9534-37daff187d9d"",""Ranked"",""Arctic Base"",40,""2024-12-31"",{8,1,7,24940,9011,91.92,26.08,6},{412,413,[{""b163b423-1574-4799-874b-f60c5943be55"",""Epic Shield"",""Rare"",1887}, {""d31d543d-d2dc-465f-83d2-607b50e7ff04"",""Mythic Ring"",""Rare"",6528}, {""41fbb686-06e4-47f8-a760-007c2f360551"",""Epic Shield"",""Mythic"",7217}]}}, … {""a12c789c-3099-47b7-9c26-1d4f8ec4cf2d"",""Ranked"",""Arctic Base"",37,""2024-11-14"",{14,9,11,7126,4369,41.58,13.58,0},{851,251,[{""8120762a-1450-4555-a628-c50b7f28dc68"",""Mythic Ring"",""Epic"",8433}, {""67a7cfa1-d0fd-47c2-b53e-a10612d39a3c"",""Mythic Ring"",""Legendary"",826}, {""dc003003-c6e8-4ee0-ada1-fbf03c0138b1"",""Legendary Armor"",""Rare"",9823}]}}]}","[{""33b25f51-088d-4014-8e23-06fd74a6c8ae"",""Pro Series 7"",2,48192,9}, {""b3c18160-022a-4d1b-9299-5d608013971b"",""Champion Series 1"",13,98075,5}, … {""7b48d01e-4255-4061-87a5-163040e8cf0a"",""Pro Series 2"",15,46796,6}]"
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""Squad Johnson Inc""","""2024-11-22""",493,498804,"{""f3c264e6-6b6c-4542-a133-380b531a43ac"",""moniquelowery"",{""joel28@example.net"",""2024-12-05"",true,""Romania"",""en""},{7,13223,609,53.76,4778,67},{{7673,83614},[{""dbd4d2ea-02df-4e19-ac5e-f8d2ad816551"",""Ancient Relic"",""Accessory"",""Rare"",86,{86,58,7,58}}, {""57629505-1959-4551-b397-0c68aeee914a"",""Ancient Relic"",""Armor"",""Common"",94,{15,54,42,54}}, … {""5aa395da-39f1-42ab-a300-c5685d39452d"",""Heavy Armor"",""Consumable"",""Common"",81,{26,47,16,58}}]},[{""5f98e39f-41af-45f3-b433-21cb0c9167db"",""Ultimate Boss Slayer"",""Hard"",29.01,695,""2024-11-09""}, {""3abc0c56-7864-4126-8518-0491712495fa"",""Master of Combat"",""Easy"",67.5,559,""2025-01-17""}, … {""2f805d28-6047-4b84-8da2-f34450b36008"",""Master of Combat"",""Hard"",48.06,562,""2024-11-07""}],[{""41a5c3b3-373d-4487-ab48-35b10eda462a"",""Ranked"",""Jungle Ruins"",31,""2024-11-05"",{18,4,12,1563,1370,44.1,14.12,6},{379,466,[{""1cf5b644-f4d3-4190-8a17-4dcb99c861e3"",""Rare Sword"",""Epic"",4072}, {""a1a5e373-abcc-4338-83e0-c21943dad942"",""Epic Shield"",""Rare"",8673}, {""35d281b5-6161-4e91-b510-78ad5389c861"",""Rare Sword"",""Rare"",2280}]}}, {""6f015e8a-9a22-46dd-851f-9daa306f809c"",""Casual"",""Underground City"",38,""2024-11-05"",{9,18,11,38965,5,52.12,22.1,9},{653,181,[{""dd5d91f9-395a-40c4-acbc-abf49afe3417"",""Legendary Armor"",""Common"",247}, {""354e010a-fb47-462a-ae03-ca993544b62a"",""Mythic Ring"",""Legendary"",1917}, {""1667ebe5-cee5-4fd7-8180-6b5a40cf3f32"",""Rare Sword"",""Common"",6888}]}}, … {""c63577b6-deb9-4265-ae2a-ecbad4b024fb"",""Casual"",""Space Station"",38,""2024-12-11"",{12,14,7,43558,6730,63.8,14.89,0},{310,206,[{""9d375b61-55d7-40cb-96c8-7af158c93c8b"",""Rare Sword"",""Rare"",3578}]}}]}","[{""33b25f51-088d-4014-8e23-06fd74a6c8

In [16]:
df_members_exploded.shape  # we exploded from 14701 -> 110613 records

(110613, 7)

In [21]:
### We can also spread (unnest) struct into multiple columns

In [17]:
df_members_exploded_spread = df_members_exploded.unnest("members")

In [18]:
df_members_exploded_spread.head(1)

team_id,name,created_date,ranking,total_winnings,player_id,username,account_details,stats,inventory,achievements,recent_matches,tournament_history
str,str,str,i64,i64,str,str,struct[5],struct[6],struct[2],list[struct[6]],list[struct[7]],list[struct[5]]
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""Squad Johnson Inc""","""2024-11-22""",493,498804,"""f16ba9c0-f9ea-4079-b206-a8e0b7…","""marvinthomas""","{""kevin49@example.com"",""2025-01-02"",true,""Gibraltar"",""ja""}","{91,159523,313,50.35,1173,32}","{{5067,85179},[{""90f6348c-3f55-46cd-8de4-66c3d1fc0ac9"",""Heavy Armor"",""Accessory"",""Epic"",87,{16,2,90,46}}, {""bf43fee5-9024-4742-bfc9-d9920efe82c8"",""Dragon Sword"",""Accessory"",""Legendary"",10,{72,29,20,74}}, … {""3f715010-385a-449b-bfc5-be92c6255517"",""Mage Staff"",""Accessory"",""Rare"",79,{94,38,54,21}}]}","[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}]","[{""7f06e144-0d2c-439f-9c98-7a0354f26ef3"",""Ranked"",""Desert Temple"",16,""2024-11-16"",{6,2,15,44202,2655,62.51,28.01,2},{515,231,[{""9e234564-ea7c-490f-993c-6242f47adf23"",""Epic Shield"",""Common"",4508}]}}, {""6643172c-a3b8-47af-9534-37daff187d9d"",""Ranked"",""Arctic Base"",40,""2024-12-31"",{8,1,7,24940,9011,91.92,26.08,6},{412,413,[{""b163b423-1574-4799-874b-f60c5943be55"",""Epic Shield"",""Rare"",1887}, {""d31d543d-d2dc-465f-83d2-607b50e7ff04"",""Mythic Ring"",""Rare"",6528}, {""41fbb686-06e4-47f8-a760-007c2f360551"",""Epic Shield"",""Mythic"",7217}]}}, … {""a12c789c-3099-47b7-9c26-1d4f8ec4cf2d"",""Ranked"",""Arctic Base"",37,""2024-11-14"",{14,9,11,7126,4369,41.58,13.58,0},{851,251,[{""8120762a-1450-4555-a628-c50b7f28dc68"",""Mythic Ring"",""Epic"",8433}, {""67a7cfa1-d0fd-47c2-b53e-a10612d39a3c"",""Mythic Ring"",""Legendary"",826}, {""dc003003-c6e8-4ee0-ada1-fbf03c0138b1"",""Legendary Armor"",""Rare"",9823}]}}]","[{""33b25f51-088d-4014-8e23-06fd74a6c8ae"",""Pro Series 7"",2,48192,9}, {""b3c18160-022a-4d1b-9299-5d608013971b"",""Champion Series 1"",13,98075,5}, … {""7b48d01e-4255-4061-87a5-163040e8cf0a"",""Pro Series 2"",15,46796,6}]"


In [19]:
df_members_exploded_spread.shape

(110613, 13)

In [25]:
### we can now access elements from structs

In [20]:
df_members_exploded_spread.select("account_details").schema

Schema([('account_details',
         Struct({'email': String, 'registration_date': String, 'premium_status': Boolean, 'country': String, 'language': String}))])

In [21]:
# extract 2 fileds from struct
df_members_exploded_spread.select(
    pl.col("account_details").struct["country"],
    pl.col("account_details").struct["premium_status"],
)

country,premium_status
str,bool
"""Gibraltar""",true
"""Romania""",true
"""Egypt""",true
"""Malta""",false
"""Antarctica (the territory Sout…",true
…,…
"""Mali""",false
"""New Zealand""",true
"""Burundi""",false


In [22]:
# extract filed from list
df_members_exploded_spread.select("achievements").schema

Schema([('achievements',
         List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})))])

In [23]:
df_members_exploded_spread.select("achievements").head(1)

achievements
list[struct[6]]
"[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}]"


In [24]:
df_members_exploded_spread.select(pl.col("achievements").list.get(0)).head(
    1
)  # first element from list

achievements
struct[6]
"{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}"


In [25]:
df_members_exploded_spread.select(
    pl.col("achievements").list.eval(pl.element().struct["points"])
)  # extract field from list of structs

achievements
list[i64]
"[662, 847, … 729]"
"[695, 559, … 562]"
"[339, 179, … 370]"
"[462, 52, … 780]"
"[293, 23, … 218]"
…
"[525, 107, … 871]"
"[513, 863, … 736]"
"[525, 681, … 410]"


In [26]:
df_members_exploded_spread.select(
    pl.col("achievements").list.eval(pl.element().struct["points"]).list.mean()
)  # calculate mean points

achievements
f64
490.875
544.090909
461.222222
442.25
435.833333
…
502.0
578.75
656.5


In [27]:
df_members_exploded_spread.select(pl.col("achievements")).explode(
    pl.col("achievements")
).unnest("achievements")

id,name,difficulty,completion_rate,points,date
str,str,str,f64,i64,str
"""5695d538-f8c2-462e-b90b-269c93…","""Ultimate Boss Slayer""","""Extreme""",45.57,662,"""2024-12-28"""
"""59e568e5-4f56-492d-8e21-100193…","""100% Completion""","""Medium""",24.42,847,"""2025-01-13"""
"""c26f4010-0162-4e96-9497-d0f651…","""Speed Runner""","""Extreme""",56.01,109,"""2025-01-09"""
"""c187c0e0-231f-4ef4-b5c3-2e9216…","""Master of Combat""","""Extreme""",31.29,836,"""2024-11-14"""
"""451377a4-498b-4fbc-8405-82f4d3…","""Speed Runner""","""Hard""",97.69,277,"""2025-01-04"""
…,…,…,…,…,…
"""821baf37-9aee-481f-9b6f-c17456…","""100% Completion""","""Hard""",82.26,755,"""2024-12-19"""
"""d1fd0e35-69b3-4457-857d-5cecc7…","""Speed Runner""","""Medium""",57.41,47,"""2024-12-31"""
"""d409dce4-439c-4e6c-bbfb-3fec15…","""Master of Combat""","""Easy""",94.94,650,"""2024-12-25"""


In [ ]:
df_members_exploded_spread.select("team_id", "player_id", "achievements").explode(
    pl.col("achievements")
).unnest("achievements").group_by("team_id", "player_id").agg(
    pl.col("completion_rate").mean().alias("avg_player_completion_rate"),
    pl.col("points").mean().alias("avg_player_points"),
)

team_id,player_id,avg_player_completion_rate,avg_player_points
str,str,f64,f64
"""7189d435-5b2d-4655-8338-8bab82…","""4143f423-d7b7-4d82-b68e-758e83…",44.590625,560.375
"""095558bd-83c6-4254-9b81-5f06c7…","""736cd0c6-628d-42d8-b61d-d7b978…",63.263,373.8
"""0fd13772-afd5-41fd-b694-60ce22…","""ca3b4611-1f89-46d0-9b2e-fda51f…",52.713333,528.5
"""3debf016-19d6-4e98-b4fc-74be2e…","""ebf2d783-6cf6-4930-bf57-119f8b…",41.27375,593.5
"""b69d1787-5f42-4aa9-bb24-a79320…","""62c7dcf9-84e9-4dea-b4e1-c1d806…",47.456316,446.736842
…,…,…,…
"""35b8550e-0210-4088-8217-f30b75…","""e4cdb1bf-f596-4a30-b901-b42e2f…",44.204,448.0
"""885ad80d-9e91-4441-927a-32de04…","""d76d9df7-17d9-4ddf-b8c5-b4bc45…",50.925625,421.1875
"""e46ecb38-560f-41bf-b96b-c75650…","""e0f4af1b-2265-4d2e-8a16-682b40…",56.056471,511.647059


In [42]:
df_members_exploded_spread.select(
    pl.col("team_id"), pl.col("player_id"), pl.col("stats")
).unnest("stats")

team_id,player_id,level,experience,total_matches,win_rate,playtime_hours,achievements_completed
str,str,i64,i64,i64,f64,i64,i64
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""f16ba9c0-f9ea-4079-b206-a8e0b7…",91,159523,313,50.35,1173,32
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""f3c264e6-6b6c-4542-a133-380b53…",7,13223,609,53.76,4778,67
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""3ca0008c-f50c-4afa-81d4-86fbae…",81,160299,710,60.64,4647,10
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""dc4b5442-2daf-452b-8de9-7e217c…",41,47191,225,49.86,1274,25
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""b81d68ff-41dd-4532-9b80-98bd5f…",19,22781,549,54.27,879,42
…,…,…,…,…,…,…,…
"""3b10d6b9-f127-4945-b1d3-202c51…","""49072ac3-925a-4fe8-877f-35edfd…",77,84700,324,46.04,2769,66
"""3b10d6b9-f127-4945-b1d3-202c51…","""7d233f9e-ff70-4bec-9c29-9831ce…",20,30340,514,42.26,3583,51
"""3b10d6b9-f127-4945-b1d3-202c51…","""a4353426-9bac-4a61-86b4-6a1567…",30,45990,126,49.89,3306,95


In [44]:
df_members_exploded_spread.head(1)

team_id,name,created_date,ranking,total_winnings,player_id,username,account_details,stats,inventory,achievements,recent_matches,tournament_history
str,str,str,i64,i64,str,str,struct[5],struct[6],struct[2],list[struct[6]],list[struct[7]],list[struct[5]]
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""Squad Johnson Inc""","""2024-11-22""",493,498804,"""f16ba9c0-f9ea-4079-b206-a8e0b7…","""marvinthomas""","{""kevin49@example.com"",""2025-01-02"",true,""Gibraltar"",""ja""}","{91,159523,313,50.35,1173,32}","{{5067,85179},[{""90f6348c-3f55-46cd-8de4-66c3d1fc0ac9"",""Heavy Armor"",""Accessory"",""Epic"",87,{16,2,90,46}}, {""bf43fee5-9024-4742-bfc9-d9920efe82c8"",""Dragon Sword"",""Accessory"",""Legendary"",10,{72,29,20,74}}, … {""3f715010-385a-449b-bfc5-be92c6255517"",""Mage Staff"",""Accessory"",""Rare"",79,{94,38,54,21}}]}","[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}]","[{""7f06e144-0d2c-439f-9c98-7a0354f26ef3"",""Ranked"",""Desert Temple"",16,""2024-11-16"",{6,2,15,44202,2655,62.51,28.01,2},{515,231,[{""9e234564-ea7c-490f-993c-6242f47adf23"",""Epic Shield"",""Common"",4508}]}}, {""6643172c-a3b8-47af-9534-37daff187d9d"",""Ranked"",""Arctic Base"",40,""2024-12-31"",{8,1,7,24940,9011,91.92,26.08,6},{412,413,[{""b163b423-1574-4799-874b-f60c5943be55"",""Epic Shield"",""Rare"",1887}, {""d31d543d-d2dc-465f-83d2-607b50e7ff04"",""Mythic Ring"",""Rare"",6528}, {""41fbb686-06e4-47f8-a760-007c2f360551"",""Epic Shield"",""Mythic"",7217}]}}, … {""a12c789c-3099-47b7-9c26-1d4f8ec4cf2d"",""Ranked"",""Arctic Base"",37,""2024-11-14"",{14,9,11,7126,4369,41.58,13.58,0},{851,251,[{""8120762a-1450-4555-a628-c50b7f28dc68"",""Mythic Ring"",""Epic"",8433}, {""67a7cfa1-d0fd-47c2-b53e-a10612d39a3c"",""Mythic Ring"",""Legendary"",826}, {""dc003003-c6e8-4ee0-ada1-fbf03c0138b1"",""Legendary Armor"",""Rare"",9823}]}}]","[{""33b25f51-088d-4014-8e23-06fd74a6c8ae"",""Pro Series 7"",2,48192,9}, {""b3c18160-022a-4d1b-9299-5d608013971b"",""Champion Series 1"",13,98075,5}, … {""7b48d01e-4255-4061-87a5-163040e8cf0a"",""Pro Series 2"",15,46796,6}]"


In [70]:
df_users = df_members_exploded_spread.select(
    pl.col("team_id"),
    pl.col("player_id"),
    pl.col("username"),
    pl.col("account_details"),
    pl.col("achievements"),
    pl.col("stats"),
)

In [74]:
df_members_exploded_spread.select("achievements").schema

Schema([('achievements',
         List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})))])

In [71]:
df_users.head()

team_id,player_id,username,account_details,achievements,stats
str,str,str,struct[5],list[struct[6]],struct[6]
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""f16ba9c0-f9ea-4079-b206-a8e0b7…","""marvinthomas""","{""kevin49@example.com"",""2025-01-02"",true,""Gibraltar"",""ja""}","[{""5695d538-f8c2-462e-b90b-269c93420b89"",""Ultimate Boss Slayer"",""Extreme"",45.57,662,""2024-12-28""}, {""59e568e5-4f56-492d-8e21-10019378d60b"",""100% Completion"",""Medium"",24.42,847,""2025-01-13""}, … {""352075ec-f2aa-4bfb-8cb4-352269962b59"",""100% Completion"",""Hard"",1.33,729,""2024-11-14""}]","{91,159523,313,50.35,1173,32}"
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""f3c264e6-6b6c-4542-a133-380b53…","""moniquelowery""","{""joel28@example.net"",""2024-12-05"",true,""Romania"",""en""}","[{""5f98e39f-41af-45f3-b433-21cb0c9167db"",""Ultimate Boss Slayer"",""Hard"",29.01,695,""2024-11-09""}, {""3abc0c56-7864-4126-8518-0491712495fa"",""Master of Combat"",""Easy"",67.5,559,""2025-01-17""}, … {""2f805d28-6047-4b84-8da2-f34450b36008"",""Master of Combat"",""Hard"",48.06,562,""2024-11-07""}]","{7,13223,609,53.76,4778,67}"
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""3ca0008c-f50c-4afa-81d4-86fbae…","""uroberson""","{""osborntimothy@example.com"",""2025-01-11"",true,""Egypt"",""ja""}","[{""b72b2382-b630-42eb-b5c2-5ccffd4a0cac"",""100% Completion"",""Easy"",42.78,339,""2024-11-11""}, {""bab23ad7-4647-40b6-9530-dede2545f3b9"",""100% Completion"",""Extreme"",12.55,179,""2024-11-04""}, … {""8a167687-e235-46df-8caf-f64bceb79c9a"",""Hidden Treasure"",""Easy"",17.88,370,""2025-01-08""}]","{81,160299,710,60.64,4647,10}"
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""dc4b5442-2daf-452b-8de9-7e217c…","""wellis""","{""lindsay48@example.com"",""2024-12-18"",false,""Malta"",""es""}","[{""319de773-636a-4841-bd7a-304d7dc8fe51"",""Master of Combat"",""Medium"",96.14,462,""2024-12-29""}, {""f24cf30e-35bb-4b05-8160-0b7952a9a4cb"",""Speed Runner"",""Medium"",81.51,52,""2024-10-28""}, … {""6c49cbcb-9bb3-413b-b6c0-5e4776a3ddde"",""Master of Combat"",""Easy"",13.25,780,""2024-11-06""}]","{41,47191,225,49.86,1274,25}"
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""b81d68ff-41dd-4532-9b80-98bd5f…","""robinsonlaura""","{""gary69@example.net"",""2025-01-12"",true,""Antarctica (the territory South of 60 deg S)"",""ja""}","[{""332f7833-6662-48cc-b24a-25c4f877f78f"",""Master of Combat"",""Hard"",80.5,293,""2024-12-01""}, {""216cae0a-3a6c-4070-ab23-8a37d36ba987"",""Ultimate Boss Slayer"",""Easy"",73.74,23,""2025-01-06""}, … {""9d10b8ad-da13-4db2-a624-259205a0cd38"",""100% Completion"",""Medium"",48.98,218,""2024-12-05""}]","{19,22781,549,54.27,879,42}"


In [72]:
df_users.explode(pl.col("achievements")).unnest("achievements").unnest(
    "account_details"
).unnest("stats").head(1)

team_id,player_id,username,email,registration_date,premium_status,country,language,id,name,difficulty,completion_rate,points,date,level,experience,total_matches,win_rate,playtime_hours,achievements_completed
str,str,str,str,str,bool,str,str,str,str,str,f64,i64,str,i64,i64,i64,f64,i64,i64
"""3d11adbe-36d0-4af4-9ac1-194ef7…","""f16ba9c0-f9ea-4079-b206-a8e0b7…","""marvinthomas""","""kevin49@example.com""","""2025-01-02""",true,"""Gibraltar""","""ja""","""5695d538-f8c2-462e-b90b-269c93…","""Ultimate Boss Slayer""","""Extreme""",45.57,662,"""2024-12-28""",91,159523,313,50.35,1173,32


In [ ]:
# Schema([('achievements', List(Struct({'id': String, 'name': String, 'difficulty': String, 'completion_rate': Float64, 'points': Int64, 'date': String})))])


df_users_agg = (
    df_users.explode(pl.col("achievements"))
    .unnest("achievements")
    .unnest("account_details")
    .unnest("stats")
    .select(
        pl.col("team_id"),
        pl.col("player_id"),
        pl.col("username"),
        pl.col("email"),
        pl.col("registration_date").str.to_date("%Y-%m-%d"),
        (datetime.date.today() - pl.col("registration_date").str.to_date("%Y-%m-%d"))
        .dt.total_days()
        .cast(pl.Int64)
        .alias("player_days_since_registration"),
        pl.when(pl.col("premium_status").eq(True))
        .then(1)
        .otherwise(0)
        .alias("is_player_premium"),
        pl.col("premium_status"),
        pl.col("country").alias("player_country"),
        pl.col("language").alias("player_language"),
        pl.col("level").alias("player_level"),
        pl.col("experience").alias("player_experience"),
        pl.col("total_matches").alias("player_total_matches"),
        pl.col("win_rate").alias("player_win_rate"),
        pl.col("playtime_hours").alias("player_playtime_hours"),
        pl.col("achievements_completed").alias("player_achievements_completed"),
        pl.col("date").alias("player_achievement_date"),
        pl.col("points").alias("player_achievement_points"),
        pl.col("completion_rate").alias("player_achievement_completion_rate"),
        pl.col("difficulty").alias("player_achievement_difficulty"),
    )
    .group_by(
        "team_id",
        "player_id",
        "username",
        "email",
        "registration_date",
        "player_days_since_registration",
        "is_player_premium",
        "premium_status",
        "player_country",
        "player_language",
        "player_level",
        "player_experience",
        "player_total_matches",
        "player_win_rate",
        "player_playtime_hours",
        "player_achievements_completed",
    )
    .agg(
        pl.col("player_achievement_date").max().alias("player_last_achievement_date"),
        pl.col("player_achievement_points")
        .mean()
        .alias("player_avg_achievement_points"),
        pl.col("player_achievement_completion_rate")
        .mean()
        .alias("player_avg_achievement_completion_rate"),
        pl.col("player_achievement_difficulty")
        .value_counts(sort=True)
        .first()
        .struct["player_achievement_difficulty"]
        .alias("player_achievement_most_common_difficulty"),
    )
)

In [133]:
df_users_agg.head()

team_id,player_id,username,email,registration_date,player_days_since_registration,is_player_premium,premium_status,player_country,player_language,player_level,player_experience,player_total_matches,player_win_rate,player_playtime_hours,player_achievements_completed,player_last_achievement_date,player_avg_achievement_points,player_avg_achievement_completion_rate,player_achievement_most_common_difficulty
str,str,str,str,date,i64,i32,bool,str,str,i64,i64,i64,f64,i64,i64,str,f64,f64,str
"""71705a03-9a48-4d47-801a-51c16a…","""631df60a-da9d-464c-9d7d-212c8d…","""nmorris""","""mdiaz@example.org""",2024-11-05,82,0,false,"""Portugal""","""ja""",31,51491,939,63.06,203,33,"""2025-01-22""",527.636364,66.512727,"""Medium"""
"""e3af9675-5506-4e83-b7cc-bdc378…","""5ae52b69-887f-40c0-8e7f-c80072…","""blanchardmatthew""","""michael96@example.org""",2024-12-16,41,1,true,"""Uruguay""","""de""",8,10344,400,52.94,242,50,"""2025-01-12""",464.2,68.698,"""Extreme"""
"""5a4e99ca-aa03-4005-94e4-caa712…","""28c1030f-e2e0-4a9b-bb4e-d5b2ba…","""sarah26""","""ohensley@example.com""",2024-12-04,53,1,true,"""Pitcairn Islands""","""ja""",63,113904,969,48.63,4822,79,"""2025-01-12""",485.333333,47.008333,"""Easy"""
"""a284cb02-df84-43b4-b953-ee0861…","""9815c1ce-03a2-4c16-9fa3-e15ca9…","""sstevens""","""glovermonica@example.net""",2024-10-30,88,1,true,"""Mongolia""","""de""",40,60160,541,59.25,3598,43,"""2025-01-23""",508.066667,67.55,"""Easy"""
"""ecf24194-c8c3-486a-8f59-4d97b0…","""3a26bb7c-6b09-415d-890e-1e5970…","""ichristensen""","""nicholas47@example.org""",2024-10-28,90,0,false,"""Finland""","""ja""",56,83608,874,45.39,448,49,"""2025-01-06""",578.125,43.058125,"""Easy"""


In [151]:
df_users_agg.group_by("team_id").agg(
    pl.col("player_id").count().alias("members_count"),
    pl.col("is_player_premium").sum().alias("members_premium_count"),
    (pl.col("is_player_premium").sum() / pl.col("player_id").count())
    .cast(pl.Float64)
    .alias("member_premium_percentage"),
    pl.col("player_days_since_registration")
    .mean()
    .cast(pl.Int64)
    .alias("member_avg_days_since_registration"),
    pl.col("player_level").mean().cast(pl.Int64).alias("member_avg_level"),
    pl.col("player_experience").mean().cast(pl.Float64).alias("member_avg_experience"),
    pl.col("player_total_matches").mean().alias("member_avg_total_matches"),
    pl.col("player_win_rate").mean().alias("member_avg_win_rate"),
    pl.col("player_playtime_hours").mean().alias("member_avg_playtime_hours"),
    pl.col("player_playtime_hours").sum().alias("member_total_playtime_hours"),
    pl.col("player_achievements_completed")
    .sum()
    .alias("member_total_achievements_completed"),
    pl.col("player_avg_achievement_points")
    .mean()
    .alias("member_avg_achievement_points"),
    pl.col("player_avg_achievement_completion_rate")
    .mean()
    .alias("member_avg_achievement_completion_rate"),
    pl.col("player_achievement_most_common_difficulty")
    .value_counts(sort=True)
    .first()
    .struct["player_achievement_most_common_difficulty"]
    .alias("member_most_common_achievement_difficulty"),
    pl.col("player_country")
    .value_counts(sort=True)
    .first()
    .struct["player_country"]
    .alias("player_country"),
    pl.col("player_language").value_counts(sort=True).first().alias("player_language"),
).head(3)

team_id,members_count,members_premium_count,member_premium_percentage,member_avg_days_since_registration,member_avg_level,member_avg_experience,member_avg_total_matches,member_avg_win_rate,member_avg_playtime_hours,member_total_playtime_hours,member_total_achievements_completed,member_avg_achievement_points,member_avg_achievement_completion_rate,member_most_common_achievement_difficulty,player_country,player_language
str,u32,i32,f64,i64,i64,f64,f64,f64,f64,i64,i64,f64,f64,str,struct[2],struct[2]
"""7547918a-4668-47b7-83ca-774367…",5,2,0.4,55,34,52688.8,631.0,53.888,3090.2,15451,246,528.189286,50.674545,"""Easy""","{""Mayotte"",1}","{""ja"",2}"
"""0e56b981-9ef5-48ba-8913-a6455d…",6,2,0.333333,38,63,93363.0,531.833333,54.261667,2624.333333,15746,238,473.871091,51.270511,"""Hard""","{""Ireland"",1}","{""fr"",3}"
"""17beb125-9e68-42b5-95ec-3a3ab6…",10,5,0.5,43,60,86039.8,551.8,55.976,1615.6,16156,492,535.668596,54.106507,"""Easy""","{""Marshall Islands"",1}","{""en"",5}"
